# Learning to Reset Demo

This notebook is the public demo path for the local pilot. It shows the result we can honestly demonstrate on CPU/MPS: raw generations fail to produce useful answers, while reset-aware retry recovers answer validity. The paper-faithful 36.94% hard-Countdown result remains the Phase 3 GPU run.

In [ ]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

def load_json(path, fallback=None):
    path = ROOT / path
    if path.exists():
        return json.loads(path.read_text())
    return fallback

raw = load_json('tmp/paper-eval/sft-hard-focus-plus-mined-fresh-eval-e0.5-hard-raw/summary.json', {
    'total_examples': 32, 'valid': 0, 'correct': 0, 'valid_rate': 0.0, 'accuracy': 0.0, 'clean_rate': 0.0
})
reset = load_json('tmp/paper-eval/sft-hard-focus-plus-mined-fresh-eval-e0.5-hard/summary.json', {
    'total_examples': 32, 'valid': 29, 'correct': 1, 'valid_rate': 29/32, 'accuracy': 1/32, 'clean_rate': 1.0
})
grounded = load_json('tmp/paper-eval/sft-grounded-26apr-seed219mine-fresh32-multiclean3/summary.json', {
    'total_examples': 32, 'valid': 31, 'correct': 0, 'valid_rate': 31/32, 'accuracy': 0.0, 'clean_rate': 1.0
})
raw, reset, grounded

In [ ]:
rows = [
    ('raw 0.5B', raw),
    ('reset-aware 0.5B', reset),
    ('grounded follow-up', grounded),
]
print('| setup | validity | hard correct | clean rate |')
print('|---|---:|---:|---:|')
for name, summary in rows:
    total = summary['total_examples']
    print(f"| {name} | {summary['valid']}/{total} | {summary['correct']}/{total} | {summary.get('clean_rate', 0):.2f} |")
print('| paper 1B reported | -- | 36.94% | -- |')

In [ ]:
results_path = ROOT / 'tmp/paper-eval/sft-grounded-26apr-seed219mine-fresh32-multiclean3/results.jsonl'
if results_path.exists():
    for line in results_path.read_text().splitlines():
        row = json.loads(line)
        if row.get('is_valid') and not row.get('reaches_target'):
            print('source:', row['source_id'])
            print('expression:', row.get('expression'))
            print('verifier value:', row.get('value'))
            print('reason:', row.get('reason'))
            print('\nresponse snippet:\n', row.get('response', '')[:900])
            break
else:
    print('Run artifacts not present. See docs/grounded-recovery-results-2026-04-26.md for the qualitative failure case.')

## What This Proves

- The local reset loop is real: raw invalid outputs can be turned into well-formed post-clean answers.
- The local 0.5B ceiling is also real: valid answers still often miss the target.
- The next honest replication step is the documented 1B+ GPU run in `docs/paper-replication.md`.